# FoMoNMR fine-tuning results

Read the existing endpoint CSVs for three fine-tuned checkpoints. This notebook does not train models or compare against the frozen benchmark, whose cohort is smaller. Run all cells in the main environment.

Ames uses test ROC-AUC from predicted probabilities; the regression endpoints use test R² (1 − squared prediction error / squared deviation from the test mean). Metrics describe individual spectrum records and are already computed in the CSVs. Mean rank averages within-endpoint ranks over all five tasks, with average ranks for ties; it does not average ROC-AUC and R². Lower rank is better. There is one run per configuration, so differences are descriptive.

ADMET removal from rich pretraining does not guarantee absence from SimNMR/NMRGym or other pretraining sources.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "dvc.yaml").is_file())
DATASETS = ["solubility_aqsoldb", "ld50_zhu", "lipophilicity_astrazeneca", "sangster_logp", "ames"]
RESULTS_DIR = PROJECT_ROOT / "results/fomonmr_finetune"
ANALYSIS_DIR = RESULTS_DIR / "analysis"
REPRESENTATIONS = ["fomonmr-pre-shifts", "fomonmr-post-rich", "fomonmr-post-unimol-rich"]


In [ ]:
frames = []
for representation in REPRESENTATIONS:
    for dataset in DATASETS:
        frame = pd.read_csv(RESULTS_DIR / representation / f"{dataset}.csv")
        assert len(frame) == 1 and frame["dataset"].eq(dataset).all(), f"Unexpected rows: {representation}/{dataset}"
        frame["representation"] = representation
        frames.append(frame)
results = pd.concat(frames, ignore_index=True)
counts = ["n_train", "n_validation", "n_test"]
assert results.groupby("dataset")[counts].nunique().eq(1).all().all(), "Different fine-tuning cohorts"
results["metric"] = np.where(results["dataset"].eq("ames"), "roc_auc", "r2")
results["score"] = np.where(results["dataset"].eq("ames"), results["roc_auc"], results["r2"])
assert np.isfinite(results["score"]).all(), "Non-finite primary metrics"
scores = results[["dataset", "representation", "run_id", "run_name", "metric", "score"] + counts].sort_values(["dataset", "representation"])
display(scores.groupby("dataset")[counts].first().reindex(DATASETS))


## Compare the three fine-tuned checkpoints

The table uses the common endpoint order used throughout the property-prediction evaluation. Original run IDs/names are retained in the long CSV; folder names provide the consistent representation labels.

In [ ]:
display(scores.pivot(index="representation", columns="dataset", values="score").reindex(
    index=REPRESENTATIONS, columns=DATASETS).round(4))
ranked = scores.copy()
ranked["rank"] = ranked.groupby("dataset")["score"].rank(ascending=False, method="average")
mean_rank = ranked.groupby("representation", as_index=False).agg(
    mean_rank=("rank", "mean"), n_endpoints=("dataset", "nunique")).sort_values(["mean_rank", "representation"])
assert mean_rank["n_endpoints"].eq(len(DATASETS)).all()
display(mean_rank)


In [ ]:
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
scores.to_csv(ANALYSIS_DIR / "scores.csv", index=False)
mean_rank.to_csv(ANALYSIS_DIR / "mean_rank.csv", index=False)
print(f"Saved scores.csv and mean_rank.csv to {ANALYSIS_DIR}")
